# Quality Checks — Encuesta
Reglas de consistencia lógica. Cada regla genera un bloque de alarma con los registros afectados.

In [ ]:
import pandas as pd
from pathlib import Path

STAGING_DIR = Path('..') / 'staging'

staging_files = sorted(
    [f for f in STAGING_DIR.glob('*.xlsx') if 'dictionary' not in f.name],
    key=lambda f: f.stat().st_mtime, reverse=True
)
if not staging_files:
    raise FileNotFoundError('No hay archivos en staging/. Corre primero pipeline.py')

df = pd.read_excel(staging_files[0])
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())
df.insert(0, 'id', ['ENC-' + str(i).zfill(4) for i in range(1, len(df) + 1)])

# Columna categorizada de Q11 (replica la misma lógica de EDA)
COL_11 = '11_experiencia_de_uso_del_seguro_en_la_atención_de_una_enfermedad_o_emergencia_colocar_las_ideas_centrales_y_palabras_claves_mencionadas_por_la_persona_encuestada_ej_atención_de_un_parto_buena_experiencia_con_el_seguro_cobertura_de_hospitalización_y_gastos_al_100_opinión_regular_sobre_la_clínica_utilizada_dificultad_en_llenar_formularios_de_reembolso_colocar_n_a_en_caso_que_la_persona_encuestada_no_haya_brindado_información_registrar_experiencias_tanto_positivas_como_negativas'

EXP_MAP = {
    'AFILIADO COMENTA QUE TUVO UNA OPERACIÓN Y SU EXPERIENCIA FUE BUENA COMENTA QUE FUERON EFICIENTES.': 'Positiva - Cirugía',
    'COMENTA QUE PARA UNA OPERACIÓN DE HIJA PARA VESÍCULA Y FUE SATISFACTORIA': 'Positiva - Cirugía',
    'recientemente su hija tuvo una operación en la cual el seguro si le cubrio una gran parte de los gastos y la atención fue buena': 'Positiva - Cirugía',
    'AFILIADO SATISFECHO CON LA ATENCIÓN RECIBIDA EN UNA HOSPITALIZACIÓN PASADA CON SU ESPOSA': 'Positiva - Hospitalización',
    'El año pasado uno de sus hermanos acudio a un hospital particular pero no le atendieron, despues al usar la cobertura de inmedical recibio atencion hospitalaria y solo tuvieron que pagar la mitad de los gastos.': 'Positiva - Hospitalización',
    'Infección Intestinal con hospitalización, fue buena la atención': 'Positiva - Hospitalización',
    'Le gusta la cobertura que recibe ya que el titular está en un tratamiento que es muy efectivo, incluso este fin de semana tiene programado una cobertura hospitalaria, la cual se la entregan rapido y sin esperar.': 'Positiva - Hospitalización',
    'LE GUSTA EL SEGURO PORQUE RECIENTEMENTE ESTUVO INTERNADA EN UNA CLINICA Y LOS GASTOS LO CUBRIÓ EL SEGURO AL IGUAL QUE LA MEDICACION': 'Positiva - Hospitalización',
    'Hospitalización por emergencia por apendicitis; la atención fue buena.': 'Positiva - Hospitalización',
    'Uno de los hijos recientemente tuvo una hospitalizacion y el seguo logró cubrir parte de los gastos médicos': 'Positiva - Hospitalización',
    'El año pasado tuvo una hospitalizacion uno de los nietos del titular y si recibio una buena atencion y cobertura de parte del seguro': 'Positiva - Hospitalización',
    'Estuvo interna por emergencia, comenta que la experiencia fue buena': 'Positiva - Hospitalización',
    'MYU BUENA TENCION EN LA ATENCION DE 3 DIAS DE HOSPITALIZADA': 'Positiva - Hospitalización',
    'BUENA ATENCION EN EL HOSPITAL': 'Positiva - Hospitalización',
    'SU ESPOSO FUE ATENDIDO EN UNA EMERGENCIA Y QUEDO MUY SATISFECHA': 'Positiva - Emergencia',
    'EN UNA EMERGENCIA QUE TUVO CON SU ESPOSO FUE BIEN ATENDIDA Y ESTA SATISFECHA': 'Positiva - Emergencia',
    'Un sobrino recibió una atención por emergencia sin tener que esperar y le cubrio la totalidad de la atención y tambien los medicamentos': 'Positiva - Emergencia',
    'FUE BIEN ATENDIDO EL PACIENTE EN EMERGENCIA': 'Positiva - Emergencia',
    'BUENA EMERGENCIA': 'Positiva - Emergencia',
    'MUY BUENA ATENCION EN EMERGENCIA': 'Positiva - Emergencia',
    'TUVO UN PARTO Y UNA ATENCIÓN POR EMERGENCIA Y FUE BIEN ATENDIDO': 'Positiva - Parto / Maternidad',
    'TUVO DOS EXPERIENCIAS DE PARTO EN LA PRIMERA SATISFECHO POR LA COBERTURA Y EN LA SEGUNDA DE LA MISMA MANERA': 'Positiva - Parto / Maternidad',
    'Su esposa estuvo embarazada y pudo cubrir los gastos utilizando la cobertura y la atencion de Plan Salud Red': 'Positiva - Parto / Maternidad',
    'ESTA SATISFECHA CON LA ATENCIÓN EN ODONTOLOGÍA YA QUE ES LA ÚNICA QUE A USADO': 'Positiva - Odontología',
    'Solo ha tenido citas en odontología y ha sido bueno el servicio': 'Positiva - Odontología',
    'UNICAMENTE FUE CON FIEBRE EN UNA OCACION Y FUE BIEN ATENDIDA': 'Positiva - Medicina general',
    'Cuando siente algun malestar, utiliza la cobertura del seguro inmedical y si le ayudan con citas médicas': 'Positiva - Medicina general',
    'ACUDIÓ POR UNA FIEBRE Y ESTA SATISFECHA': 'Positiva - Medicina general',
    'recientemente ha tenido citas médicas y si le entregan el servicio de manera rapida y satisfactoria': 'Positiva - Medicina general',
    'Le gusta que el seguro ademas de las citas medicas tambien le da cobertura para realizar examenes médicos que son ordenados por el doctor': 'Positiva - General',
    'Hasta ahora es muy buena la cobertura, le han dado atencion cuando lo necesitan y le dan cobertura para examenes médicos': 'Positiva - General',
    'COMENTA QUE HABLANDO DE EXÁMENES NO TIENE COBERTURA PARA RX O LABORATORIO': 'Negativa - Cobertura insuficiente',
    'NO CUBRIA EL SEGURO EL PROCEDIMIENTO QUE SE REALIZO': 'Negativa - Cobertura insuficiente',
    'EL SEGURO NO CUBRIO NADA DE LA CIRUGIA DEL ESPOSO DE LA TITULAR': 'Negativa - Cobertura insuficiente',
    'NO CUBRIO EL SEGURO LA OPERACION DE SU HIJO, NO LE REALIZARON EL REMBOLSO.': 'Negativa - Cobertura insuficiente',
    'NO CUBRE EL MEDICAMENTO NI TIENEN TODAS LAS ESPECIALIDADES': 'Negativa - Cobertura insuficiente',
    'NO CUBRE NADA EL SEGURO': 'Negativa - Cobertura insuficiente',
    'AFILIADA COMENTA QUE NO ESTA DE ACUERDO CON QUE TENGA QUE TENER DERIVACIÓN PARA ALUNA ESPECIALIDAD YA QUE LE TOMA TIEMPO RECIBIR LA SIGUIENTE CITA MEDICA.': 'Negativa - Acceso a especialidades',
    'NO ATIENDEN ESPECIALISTAS, EL MISMO MEDICO ATIENDE A TODOS': 'Negativa - Acceso a especialidades',
    'NO TIENE BUENA ATENCIÓN, YA QUE HASTA POR QUERER ODONTOLOGÍA DEBE PASAR POR MG': 'Negativa - Acceso a especialidades',
    'AFILIADA COMENTA QUE A SOLICITADO CITAS MEDICAS Y NO LE HAN AGENDADO POR FALTA DE PAGO CON LOS CENTROS MÉDICOS QUE ESTÁN CERCANOS, USA POCO EL SEGURO DE INMEDICAL.': 'Negativa - Atención deficiente',
    'ATENCIÓN EN MEDICINA GENERAL SOCIO CONSIDERA QUE FUE POCO SATISFACTORIA ADEMAS QUE LA MEDICINA FUE POCO CUBIERTA PARA LO QUE LE ENVIARON Y NECESITA': 'Negativa - Atención deficiente',
    'OBTUVO UNA MALA EXPERIENCIA AL ACUDIR POR SU ESPOSO PERO NO FUE INTERNADO SITIO QUE NO FUE BIEN ATENDIDA TUVO QUE SOLICITAR QUE LE HICIERAN DEVOLUCIÓN  ECONÓMICA YA QUE LE TOCO SER ATENDIDO E OTRO LUGAR. A PARTE DE ESO SATISFECHA CON LA ATENCIÓN': 'Negativa - Atención deficiente',
    'EN EMERGENCIA LA ATENCION NO FUE BUENA': 'Negativa - Atención deficiente',
    'COMENTA QUE ACUDIÓ POR EMERGENCIA CON SIGNOS GRIPALES Y ESTUVO SATISFECHO CON LA COBERTURA DE LA MEDICINA Y LA ATENCIÓN.( COMENTA QUE EN LA FAMILIA TIENEN UNA ENFERMEDAD DE DIABETES TIPO 2 CON LA QUE NO CUENTAN CON LA COBERTURA EN MEDICAMENTOS Y DEMÁS COSAS QUE NECESITAN, POCO SATISFECHOS POR ESE LADO.': 'Mixta',
    'NO TENIA COBERTURA PARA UNA RUPTURA DE TIBIA QUE SUFRIÓ SU HIJO, DE AHÍ COMENTA QUE LA ATENCIÓN ES BUENA': 'Mixta',
    'AFILIADA DIO A LUZ CONTANDO CON EL SEGURO, COMENTA QUE FUE UNA EMERGENCIA QUE NO ESTUVO CUBIERTA EN SU TOTALIDAD Y TUVO QUE CUBRIR LA AFILIADA SIENDO DESPUÉS DEVUELTO POR EL SEGURO': 'Mixta',
    'Le gusta la cobertura que brinda el seguro pero en especialidades no le gusta tener que esperar varios días para recibir una atencion en alguna especialidad': 'Mixta',
    'SU ESPOSA FUE OPERADA Y ESTA SATISFECHO CON LA COBERTURA Y ATENCIÓN.. PERO TAMBIÉN EN CLÍNICA ROMERO COMENTA QUE NO LE GUSTO LA ATENCIÓN.': 'Mixta',
    'BUENA EXPERIENCIA EN MG, MALA EXPERIENCIA EN ODONTOLOGIA Y MAL TRATO DEL MEDICO': 'Mixta',
    'LA COBERTURA DEBERIA SER MAS AMPLIA EN MEDICAMENTOS, LA ATENCION ES BUENA': 'Mixta',
    'SIN NOVEDADES EN LA ATENCION': 'Sin novedad',
    'SIN NOVEDADES': 'Sin novedad',
    'SIN NOVEDAD, TODO MUY BIEN': 'Sin novedad',
    'SIN NOVEDAD, BUENA ATENCION': 'Sin novedad',
}
POSITIVA_GENERAL_KEYWORDS = [
    'BUENA', 'BUEN', 'MUY BIEN', 'EXCELENTE', 'SATISFECH', 'BIEN ATEND',
    'NINGUNA EN PARTICULAR', 'NO TIENE QUEJAS', 'BUENAS EXPERIENCIAS',
    'BUENA INFORMACION', 'RECIBIO BUENOS', 'TELEMEDICA',
]

def categorize_exp(val):
    if pd.isna(val) or str(val).strip().upper() in ('N/A', 'NA', ''):
        return 'Sin novedad'
    v = str(val).strip()
    if v in EXP_MAP:
        return EXP_MAP[v]
    if any(kw in v.upper() for kw in POSITIVA_GENERAL_KEYWORDS):
        return 'Positiva - General'
    return 'Sin clasificar'

df['11_experiencia_categoria'] = df[COL_11].apply(categorize_exp)

print(f'Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Archivo: {staging_files[0].name}')

## Valores únicos — referencia para calibrar las reglas
Corre esta celda primero para ver los valores reales de las columnas involucradas en las reglas.

In [ ]:
COLS_RELEVANTES = {
    'Q2  Rango etario'        : '2_rango_etario_seleccionar_desde_la_base_de_datos',
    'Q5  Actividad económica' : '5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar',
    'Q6  Ingreso mensual'     : '6_ingreso_promedio_mensual_tomando_como_referencia_500_usd',
    'Q7  Nº asegurados'       : '7_no_de_personas_aseguradas_en_el_núcleo_familiar_escribir_número_sin_espacios_ej_4',
    'Q8  Grupos prioritarios' : '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado',
    'Q9  Uso plan_salud_red'     : '9_uso_del_plan_salud_red_en_el_último_año',
    'Q13 Percepción salud'    : '13_percepción_del_estado_de_salud_del_afiliado_a',
    'Q14 Gasto en salud'      : '14_gasto_en_salud_tomando_en_cuenta_el_uso_del_plan_salud_red',
    'Q15 Prom. gasto salud'   : '15_promedio_mensual_de_presupuesto_gastado_en_salud_en_el_núcleo_familiar',
}

for label, col in COLS_RELEVANTES.items():
    vals = df[col].dropna().unique()
    print(f'\n{"─"*60}')
    print(f'  {label}')
    print(f'{"─"*60}')
    for v in sorted(vals, key=str):
        n = (df[col] == v).sum()
        print(f'  [{n:>3}]  {v}')

## Definición de reglas de consistencia
⚠️ **Ajusta los valores en el bloque `CALIBRATION` si los únicos de arriba no coinciden con los esperados.**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CALIBRATION — ajusta estos valores según los únicos que imprimió la celda anterior
# ══════════════════════════════════════════════════════════════════════════

# Rangos etarios que corresponden a personas jóvenes (≤ ~40 años).
# Un jubilado en estos rangos es sospechoso.
RANGOS_JOVENES = [
    '18-25', '18 - 25', '26-30', '26 - 30',
    '31-35', '31 - 35', '36-40', '36 - 40',
]

# Rangos etarios que corresponden a personas mayores (≥ 50 años).
# Compatibles con "Jubilado".
RANGOS_MAYORES_JUBILADO = [
    '46-50', '46 - 50', '51-55', '51 - 55',
    '56-60', '56 - 60', '61-65', '61 - 65',
    'Más de 65', 'mas de 65', '> 65',
]

# Orden de los rangos de ingreso mensual de menor a mayor.
# Ajusta las claves exactas según lo que imprima la celda de únicos.
INCOME_ORDER = [
    'Menos de 470',
    'Entre 470 y 800',
    'Entre 800 y 1500',
    'Más de 1500',
]

# Orden de los niveles de gasto mensual en salud de menor a mayor.
SPENDING_ORDER = [
    'Menos de 50',
    'Entre 50 y 100',
    'Entre 100 y 200',
    'Más de 200',
]

# Percepciones de salud consideradas "positivas"
PERCEPCION_BUENA = ['Buena', 'Muy buena', 'Excelente', 'buena', 'muy buena', 'excelente']

# Percepciones de salud consideradas "negativas"
PERCEPCION_MALA  = ['Mala', 'Muy mala', 'Regular', 'mala', 'muy mala', 'regular']

# Prefijos de categorías de experiencia negativas (Q11)
EXP_NEGATIVA_PREFIX = 'Negativa'

# Prefijos de categorías de experiencia positivas (Q11)
EXP_POSITIVA_PREFIX = 'Positiva'

# Threshold de nº asegurados considerado inusualmente alto
MAX_ASEGURADOS = 8

# Columnas clave (alias)
C_RANGO   = '2_rango_etario_seleccionar_desde_la_base_de_datos'
C_ACT     = '5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar'
C_INGRESO = '6_ingreso_promedio_mensual_tomando_como_referencia_500_usd'
C_NASEG   = '7_no_de_personas_aseguradas_en_el_núcleo_familiar_escribir_número_sin_espacios_ej_4'
C_GRUPOS  = '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado'
C_USO     = '9_uso_del_plan_salud_red_en_el_último_año'
C_EXP_RAW = COL_11
C_EXP_CAT = '11_experiencia_categoria'
C_PERC    = '13_percepción_del_estado_de_salud_del_afiliado_a'
C_GASTO   = '14_gasto_en_salud_tomando_en_cuenta_el_uso_del_plan_salud_red'
C_PROM_G  = '15_promedio_mensual_de_presupuesto_gastado_en_salud_en_el_núcleo_familiar'

print('✓ Calibración cargada.')

In [ ]:
import re

# ══════════════════════════════════════════════════════════════════════════
# Motor de chequeo
# ══════════════════════════════════════════════════════════════════════════

results = []   # lista de dicts: {rule_id, description, severity, n, ids}

def _register(rule_id, description, severity, flagged_df, cols_to_show):
    """Guarda el resultado de una regla y lo imprime."""
    n = len(flagged_df)
    icon = '🔴' if severity == 'Alta' else ('🟡' if severity == 'Media' else '🔵')
    status = f'{icon} [{severity}]  {rule_id} — {description}  →  {n} registros'
    results.append({
        'rule_id'     : rule_id,
        'description' : description,
        'severity'    : severity,
        'n_flagged'   : n,
        'detail'      : flagged_df[['id'] + cols_to_show].copy() if n > 0 else pd.DataFrame(),
    })
    if n > 0:
        print(status)
    else:
        print(f'✅  {rule_id} — {description}  →  OK')


# ──────────────────────────────────────────────────────────────────────────
# R01 · Jubilado en rango etario joven
# ──────────────────────────────────────────────────────────────────────────
mask_jubilado = df[C_ACT].str.contains('Jubilado', na=False, case=False)
mask_joven    = df[C_RANGO].isin(RANGOS_JOVENES)
_register('R01', 'Jubilado en rango etario joven (≤ 40 años)', 'Alta',
          df[mask_jubilado & mask_joven], [C_RANGO, C_ACT])

# ──────────────────────────────────────────────────────────────────────────
# R02 · Nº de asegurados inválido (0, negativo, o no numérico)
# ──────────────────────────────────────────────────────────────────────────
def _parse_int(val):
    try:
        return int(str(val).strip())
    except (ValueError, TypeError):
        return None

n_aseg_num = df[C_NASEG].apply(_parse_int)
mask_invalido = n_aseg_num.isna() | (n_aseg_num <= 0)
_register('R02', 'Nº asegurados inválido (0, negativo o no numérico)', 'Alta',
          df[mask_invalido], [C_NASEG])

# ──────────────────────────────────────────────────────────────────────────
# R03 · Nº de asegurados inusualmente alto
# ──────────────────────────────────────────────────────────────────────────
mask_alto = n_aseg_num > MAX_ASEGURADOS
_register('R03', f'Nº asegurados > {MAX_ASEGURADOS} (inusual)', 'Media',
          df[mask_alto], [C_NASEG, C_ACT, C_INGRESO])

# ──────────────────────────────────────────────────────────────────────────
# R04 · "No Aplica" en Q8 combinado con otros grupos prioritarios
# ──────────────────────────────────────────────────────────────────────────
mask_no_aplica = df[C_GRUPOS].str.contains('No Aplica', na=False, case=False)
mask_otros_grp = df[C_GRUPOS].str.contains(
    'discapacidad|adultos mayores|niños|enfermedades crónicas|embarazadas',
    na=False, case=False
)
_register('R04', '"No Aplica" en Q8 combinado con otros grupos prioritarios', 'Alta',
          df[mask_no_aplica & mask_otros_grp], [C_GRUPOS])

# ──────────────────────────────────────────────────────────────────────────
# R05 · Q8 tiene "Niños <5 años" pero solo 1 asegurado
# ──────────────────────────────────────────────────────────────────────────
mask_ninos = df[C_GRUPOS].str.contains('iños', na=False, case=False)  # Niños / niños
mask_uno   = n_aseg_num == 1
_register('R05', 'Q8 incluye "Niños <5 años" pero nº asegurados = 1', 'Media',
          df[mask_ninos & mask_uno], [C_NASEG, C_GRUPOS])

# ──────────────────────────────────────────────────────────────────────────
# R06 · No usó seguro (Q9=No) pero Q11 describe experiencia concreta
# ──────────────────────────────────────────────────────────────────────────
mask_no_uso = df[C_USO].str.strip().str.lower().isin(['no'])
mask_exp_concreta = ~df[C_EXP_CAT].isin(['Sin novedad', 'Sin clasificar'])
_register('R06', 'Q9=No pero Q11 describe experiencia concreta de uso', 'Media',
          df[mask_no_uso & mask_exp_concreta], [C_USO, C_EXP_CAT, C_EXP_RAW])

# ──────────────────────────────────────────────────────────────────────────
# R07 · Percepción de salud positiva pero experiencia de uso muy negativa
# ──────────────────────────────────────────────────────────────────────────
mask_perc_buena = df[C_PERC].isin(PERCEPCION_BUENA)
mask_exp_neg    = df[C_EXP_CAT].str.startswith(EXP_NEGATIVA_PREFIX, na=False)
_register('R07', 'Percepción de salud buena pero experiencia de uso Negativa', 'Baja',
          df[mask_perc_buena & mask_exp_neg], [C_PERC, C_EXP_CAT])

# ──────────────────────────────────────────────────────────────────────────
# R08 · Percepción de salud mala/regular pero experiencia de uso muy positiva
# ──────────────────────────────────────────────────────────────────────────
mask_perc_mala = df[C_PERC].isin(PERCEPCION_MALA)
mask_exp_pos   = df[C_EXP_CAT].str.startswith(EXP_POSITIVA_PREFIX, na=False)
_register('R08', 'Percepción de salud mala/regular pero experiencia de uso Positiva', 'Baja',
          df[mask_perc_mala & mask_exp_pos], [C_PERC, C_EXP_CAT])

# ──────────────────────────────────────────────────────────────────────────
# R09 · Gasto mensual en salud mayor al ingreso mensual
#        Requiere que los valores de INCOME_ORDER y SPENDING_ORDER
#        estén correctamente calibrados arriba.
# ──────────────────────────────────────────────────────────────────────────
income_rank  = {v: i for i, v in enumerate(INCOME_ORDER)}
spend_rank   = {v: i for i, v in enumerate(SPENDING_ORDER)}

df['_income_rank'] = df[C_INGRESO].map(income_rank)
df['_spend_rank']  = df[C_PROM_G].map(spend_rank)

# Solo evalúa filas donde ambos valores están mapeados
mask_ranked  = df['_income_rank'].notna() & df['_spend_rank'].notna()
# Flag: gasto tiene rango mayor que el ingreso (sospechoso — puede ser error o deuda)
mask_gasto_alto = df['_spend_rank'] > df['_income_rank']
_register('R09', 'Gasto mensual en salud supera el ingreso mensual declarado', 'Media',
          df[mask_ranked & mask_gasto_alto], [C_INGRESO, C_PROM_G])

df.drop(columns=['_income_rank', '_spend_rank'], inplace=True)

# ──────────────────────────────────────────────────────────────────────────
# R10 · Filas completamente duplicadas (excluyendo id y marca_temporal)
# ──────────────────────────────────────────────────────────────────────────
cols_sin_id = [c for c in df.columns if c not in ('id', 'marca_temporal')]
mask_dup = df.duplicated(subset=cols_sin_id, keep=False)
_register('R10', 'Respuestas duplicadas (mismos valores en todas las columnas)', 'Alta',
          df[mask_dup], [C_RANGO, C_ACT, C_INGRESO])

print(f'\n{"═"*60}')
print(f'  Total reglas evaluadas : {len(results)}')

## Resumen ejecutivo + detalle de alertas

In [ ]:
# ── Resumen ejecutivo ────────────────────────────────────────────────────
summary = pd.DataFrame([{
    'Regla'      : r['rule_id'],
    'Descripción': r['description'],
    'Severidad'  : r['severity'],
    'N_alertas'  : r['n_flagged'],
    'Estado'     : '⚠️  ALERTA' if r['n_flagged'] > 0 else '✅  OK',
} for r in results])

total_alertas = summary['N_alertas'].sum()
n_reglas_ok   = (summary['N_alertas'] == 0).sum()

print(f'{"═"*65}')
print(f'  RESUMEN DE CALIDAD   |   {len(df)} encuestas   |   {len(results)} reglas')
print(f'{"═"*65}')
print(f'  Reglas sin alertas   : {n_reglas_ok} / {len(results)}')
print(f'  Total registros con alerta : {total_alertas}')
print(f'{"═"*65}\n')

display(summary.style.apply(
    lambda col: [
        'background-color: #FFCCCC' if v == '⚠️  ALERTA' else 'background-color: #CCFFCC'
        for v in col
    ], subset=['Estado']
))

In [ ]:
# ── Detalle de cada alerta ────────────────────────────────────────────────
SEV_ORDER = {'Alta': 0, 'Media': 1, 'Baja': 2}
alerts = sorted([r for r in results if r['n_flagged'] > 0],
                key=lambda r: SEV_ORDER.get(r['severity'], 9))

if not alerts:
    print('🎉  Sin alertas. Todos los registros pasan las reglas de consistencia.')
else:
    for r in alerts:
        icon = '🔴' if r['severity'] == 'Alta' else ('🟡' if r['severity'] == 'Media' else '🔵')
        print(f'\n{icon}  {r["rule_id"]} [{r["severity"]}] — {r["description"]}')
        print(f'   {r["n_flagged"]} registro(s) afectado(s):')
        display(r['detail'].reset_index(drop=True))